# TunBERT + Doc2Vec — Combined Embedding Topic Model

**Method:** Concatenates **TunBERT** contextual embeddings with **Doc2Vec** document embeddings trained on this corpus, compresses with an autoencoder, and trains **CombinedTM** — both across a topic-count grid search and at a fixed 6-topic configuration for closer inspection.

**Why this method:** Doc2Vec (Paragraph Vectors) learns a dense representation for each *whole document* directly from this corpus's word co-occurrence patterns, rather than relying on a pretrained model's general language knowledge. Pairing it with TunBERT tests whether corpus-specific document semantics add anything beyond what a pretrained dialect model already captures.

**Pipeline:** preprocess → TunBERT embeddings → train Doc2Vec on this corpus → concatenate with UMAP-reduced TunBERT → autoencoder-compress to 64d → CombinedTM (grid search + a closer look at 6 topics).

## 0. Setup

In [ ]:
!pip install numpy==1.26.4 gensim==4.3.3 --force-reinstall --upgrade
!pip install transformers sentence-transformers umap-learn hdbscan scikit-learn nltk pandas openpyxl contextualized-topic-models tensorflow


## 1. Load the corpus

We start from the raw Tunisian dialect social media corpus. Each row is one post/message; we drop empty rows since they carry no signal for topic modeling.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_excel("../data/TunTap_Corpus.xlsx")

# Drop rows with missing text
df = df.dropna(subset=["message"])

# Reset index
df = df.reset_index(drop=True)

# Extract just the text column
message = df["message"].tolist()


## 2. Preprocessing

Tunisian dialect on social media mixes Arabic script, Arabizi (Latin transliteration with digits standing in for Arabic letters, e.g. `3` for `ع`), French loanwords, elongated words for emphasis (`hhhhh`, `mriiiigla`), and noise (URLs, mentions, hashtags, emojis). Standard NLP preprocessing pipelines aren't built for this, so we apply dialect-specific cleaning:

1. Remove custom Tunisian stopwords (functional words with no topical meaning)
2. Normalize character elongation (`mriiiigla` → `mrigla`)
3. Convert Arabizi digits back to their Arabic-letter equivalent (`3` → `a`, `7` → `h`, `5` → `kh`, `9` → `k`, `2` → `a`)
4. Strip URLs, mentions, hashtags, emojis and non-alphanumeric symbols
5. Normalize Arabic letter variants (e.g. `إأآا` → `ا`) so the same word isn't split across multiple spellings
6. Remove stopwords a second time (some appear only after cleaning) and drop any resulting empty lines

In [ ]:
# Read stopwords.txt into a set
with open("../data/tunisian_stopwords.txt", "r", encoding="utf-8") as f:
    all_stopwords = set(line.strip() for line in f if line.strip())


In [ ]:
def remove_custom_stopwords(text, stopwords_set):
    tokens = text.split()
    filtered = [word for word in tokens if word not in stopwords_set]
    return " ".join(filtered)


In [ ]:
text = [remove_custom_stopwords(t, all_stopwords) for t in message]


In [ ]:
import re

def normalize_elongation(text):
    # Replace 2 or more repeated characters with 1
    return re.sub(r'(.)\1{1,}', r'\1', text)

def convert_tunisian_numbers(text):
    return (
        text.replace("3", "a")
            .replace("7", "h")
            .replace("5", "kh")
            .replace("9", "k")
            .replace("2", "a")
    )

def remove_numbers(text):
    # Remove all digits (0-9)
    return re.sub(r'\d+', '', text)

def clean_dual_script_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove mentions and hashtags
    text = re.sub(r"@\w+|#\w+", "", text)

    # Remove emojis and symbols (non-alphanum + Arabic letters + space)
    text = re.sub(r"[^\u0621-\u063A\u0641-\u064A\w\s]", " ", text)

    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    # convert numbers like 3 → ع
    text = convert_tunisian_numbers(text)

    # Normalize elongation
    text = normalize_elongation(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    text = remove_numbers(text)

    return text


In [ ]:
cleaned_texts = [clean_dual_script_text(t) for t in text]

In [ ]:
final_texts = [remove_custom_stopwords(t, all_stopwords) for t in cleaned_texts]

In [ ]:
final_texts = [line for line in final_texts if line.strip() != '']


Check how many documents survived preprocessing:

In [ ]:
print(len(final_texts))


17287


## 4. TunBERT embeddings

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
import umap

tokenizer = AutoTokenizer.from_pretrained("not-lain/TunBERT", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("not-lain/TunBERT", trust_remote_code=True)
model.eval()

def embed_documents(docs, batch_size=32):
    embeddings = []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    with torch.no_grad():
        for i in range(0, len(docs), batch_size):
            batch = docs[i:i+batch_size]
            for doc in batch:
                inputs = tokenizer(doc, return_tensors="pt", truncation=True, padding=True).to(device)
                outputs = model.BertModel(**inputs, output_hidden_states=True)
                cls_embedding = outputs.last_hidden_state[:, 0, :]
                embeddings.append(cls_embedding.squeeze().cpu().numpy())
    return np.array(embeddings)

print("Embedding documents with TunBERT...")
embeddings = embed_documents(final_texts)
print("Done embeddings. Shape:", embeddings.shape)


Embedding documents with TunBERT...
Done embeddings. Shape: (17287, 768)


## 3. Topic coherence utilities

We evaluate topic quality with two complementary metrics:
- **C_V coherence** — measures how semantically related the top words of a topic are, based on word co-occurrence in a sliding window (via `gensim`)
- **NPMI** (Normalized Pointwise Mutual Information) — a simpler, more interpretable co-occurrence measure computed directly on document sets, less sensitive to corpus size than raw PMI

Both are computed from `final_texts`, split into tokens.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from itertools import combinations
from collections import Counter
import math

tokenized_docs = [doc.split() for doc in final_texts]
dictionary = Dictionary(tokenized_docs)

def compute_cv_score(topics_list, tokenized_docs, dictionary):
    cm = CoherenceModel(topics=topics_list, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    return cm.get_coherence()

def compute_manual_npmi(topics_list, tokenized_docs):
    doc_sets = [set(d) for d in tokenized_docs]
    num_docs = len(doc_sets)
    word_doc_counts = Counter()
    for s in doc_sets:
        for w in s:
            word_doc_counts[w] += 1
    def topic_npmi(topic_words):
        vals = []
        for w1, w2 in combinations(topic_words, 2):
            p1 = word_doc_counts.get(w1, 0) / num_docs
            p2 = word_doc_counts.get(w2, 0) / num_docs
            co = sum(1 for s in doc_sets if w1 in s and w2 in s)
            p12 = co / num_docs
            if p12 > 0 and p1 > 0 and p2 > 0:
                pmi = math.log(p12 / (p1 * p2))
                npmi = pmi / (-math.log(p12))
                vals.append(npmi)
        return (sum(vals) / len(vals)) if vals else 0.0
    scores = [topic_npmi(topic) for topic in topics_list]
    return sum(scores) / len(scores), scores


## 5. Train Doc2Vec on this corpus and build the hybrid embedding

Unlike TunBERT and SBERT/E5, Doc2Vec isn't pretrained — it's trained from scratch here on `final_texts`, so its document vectors are shaped entirely by this corpus's own vocabulary and co-occurrence patterns.

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

tagged_data = [TaggedDocument(words=_d.split(), tags=[str(i)]) for i, _d in enumerate(final_texts)]
doc2vec_model = Doc2Vec(vector_size=200, window=10, min_count=5, epochs=200, sample=1e-5, workers=4)
doc2vec_model.build_vocab(tagged_data)
doc2vec_model.train(tagged_data, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)
doc2vec_embs = np.array([doc2vec_model.dv[str(i)] for i in range(len(tagged_data))])

# Reduce TunBERT (768d) to 128d, then combine with Doc2Vec (200d)
umap_model = umap.UMAP(n_components=128, random_state=42)
tunbert_reduced = umap_model.fit_transform(embeddings)

hybrid_embs = np.concatenate([doc2vec_embs, tunbert_reduced], axis=1)


## 6. Compress with an autoencoder

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = hybrid_embs.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(hybrid_embs, hybrid_embs, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs = encoder.predict(hybrid_embs)


## 7. Build the CTM dataset

In [ ]:
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.models.ctm import CombinedTM

tp = TopicModelDataPreparation("not-lain/TunBERT")

training_dataset = tp.fit(
    text_for_contextual=final_texts,
    text_for_bow=final_texts,
    custom_embeddings=compressed_embs
)

bow_size = len(tp.vocab)
contextual_size = compressed_embs.shape[1]


## 8. Grid search over topic counts

In [ ]:
# ===============================
# Grid search over topic counts
# ===============================
# Trains a fresh CombinedTM for each candidate number of topics and scores it,
# so we can pick the topic count that yields the most coherent topics.
topic_numbers = [5, 6, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 95, 100]
results = []

for n_topics in topic_numbers:
    print(f"\n🌀 Training CombinedTM with {n_topics} topics...")

    ctm = CombinedTM(
        bow_size=bow_size,
        contextual_size=contextual_size,
        n_components=n_topics,
        num_epochs=5,
        batch_size=64,
        hidden_sizes=(128,),
        activation="relu",
        dropout=0.0,
        lr=2e-3
    )

    ctm.fit(training_dataset)

    topics_list = ctm.get_topic_lists(20)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics_list
    ]

    texts_tokenized = [doc.split() for doc in final_texts]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")

    results.append((n_topics, cv_score, avg_npmi))

# ===============================
# Display results as table
# ===============================
results_df = pd.DataFrame(results, columns=["n_topics", "CV", "NPMI"])
print("\n📊 Grid Search Results:")
print(results_df)



🌀 Training CombinedTM with 5 topics...

✅ 5 topics → CV: 0.552, NPMI: 0.673

🌀 Training CombinedTM with 6 topics...

✅ 6 topics → CV: 0.623, NPMI: 0.549

🌀 Training CombinedTM with 7 topics...

✅ 7 topics → CV: 0.483, NPMI: 0.598

🌀 Training CombinedTM with 10 topics...

✅ 10 topics → CV: 0.508, NPMI: 0.490

🌀 Training CombinedTM with 15 topics...

✅ 15 topics → CV: 0.560, NPMI: 0.583

🌀 Training CombinedTM with 20 topics...

✅ 20 topics → CV: 0.524, NPMI: 0.607

🌀 Training CombinedTM with 25 topics...

✅ 25 topics → CV: 0.539, NPMI: 0.531

🌀 Training CombinedTM with 30 topics...

✅ 30 topics → CV: 0.547, NPMI: 0.562

🌀 Training CombinedTM with 35 topics...

✅ 35 topics → CV: 0.568, NPMI: 0.580

🌀 Training CombinedTM with 40 topics...

✅ 40 topics → CV: 0.568, NPMI: 0.589

🌀 Training CombinedTM with 45 topics...

✅ 45 topics → CV: 0.530, NPMI: 0.620

🌀 Training CombinedTM with 50 topics...

✅ 50 topics → CV: 0.508, NPMI: 0.624

🌀 Training CombinedTM with 55 topics...

✅ 55 topics → CV

## 9. Closer look at the 6-topic model

6 topics scored the best C_V coherence in the grid search above, so we retrain at that exact configuration and print the actual topics — moving from "how coherent are the topics" to "what are the topics".

In [ ]:
ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=contextual_size,
    n_components=6,
    num_epochs=5,
    batch_size=64,
    hidden_sizes=(128,),
    activation="relu",
    dropout=0.0,
    lr=2e-3
)

ctm.fit(training_dataset)

topics_list = ctm.get_topic_lists(20)
for i, topic in enumerate(topics_list):
    print(f"🟢 Topic {i+1}: {', '.join(topic)}")

topics_clean = [
    [w if isinstance(w, str) else w[0] for w in topic]
    for topic in topics_list
]

texts_tokenized = [doc.split() for doc in final_texts]
dictionary = Dictionary(texts_tokenized)
corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

print(f"✅ 6 topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")


🟢 Topic 1: صعبه, bled, alah, عملاق, labed, ti, lkol, ili, chab, tounes, tounis, winou, hata, walh, flous, chkoun, entouma, lil, yarhmou, الكبار
🟢 Topic 2: يجب, ليبيا, تجار, تبدا, حسين, الاخبار, فساد, تطبيق, يجي, لابد, مدير, الثوره, المفروض, السب, الخوانجيه, البرلمان, الجمهوريه, وقع, القديمه, الفاسد
🟢 Topic 3: ديموقراطيه, الوالد, مصرين, والاقصاء, بالقتل, فاقوش, بلديات, عاشقه, مخلالهم, اتحبوا, مخالفه, تسمعكم, الجريدي, الاحتجاجات, يتصدي, نزيدو, يرحمو, الملحمه, يرحمه, المنشط
🟢 Topic 4: الشعب, عليهم, الناس, البلاد, سعيد, تونس, قيس, ريس, لازم, الدوله, السيد, المواطن, النهضه, دوله, القانون, البرلمان, التونسي, حاجه, شعب, العام
🟢 Topic 5: البصل, يعطك, عميل, فاهم, هز, الجمهور, مشا, المجمد, يشد, عوده, تستاهلو, المشكل, كاس, استاذه, حره, دجاج, احلي, الجمعيه, تروح, سياسي
🟢 Topic 6: قالتلكم, ضربني, فالح, سروال, حالين, كليه, بالراي, سبقني, المدرسيه, حيط, احسنت, شكا, لدوله, بدي, العين, الكازي, صدقتك, كذب, موازيه, عزرايل
✅ 6 topics → CV: 0.623, NPMI: 0.549


## 10. Document-topic distribution

For a fuller picture beyond just the top words, we look at how many documents fall under each topic's dominant assignment.

In [ ]:
import numpy as np
import pandas as pd

# 1. Get document-topic probabilities
doc_topic_probs = ctm.get_doc_topic_distribution(training_dataset)

# 2. Find dominant topic for each document
dominant_topics = np.argmax(doc_topic_probs, axis=1)

# 3. Count documents per topic
topic_counts = pd.Series(dominant_topics).value_counts().sort_index()

# 4. Top 20 words per topic
topics_list = ctm.get_topic_lists(20)

# 5. Combine into a summary table
topic_summary = pd.DataFrame({
    'Topic': range(len(topics_list)),
    'Top Words': [", ".join(words) for words in topics_list],
    'Doc Count': [topic_counts.get(t, 0) for t in range(len(topics_list))]
})

print(topic_summary)


   Topic                                          Top Words  Doc Count
0      0  صعبه, bled, alah, عملاق, labed, ti, lkol, ili,...       2401
1      1  يجب, ليبيا, تجار, تبدا, حسين, الاخبار, فساد, ت...       3095
2      2  ديموقراطيه, الوالد, مصرين, والاقصاء, بالقتل, ف...       3235
3      3  الشعب, عليهم, الناس, البلاد, سعيد, تونس, قيس, ...       2128
4      4  البصل, يعطك, عميل, فاهم, هز, الجمهور, مشا, الم...       4858
5      5  قالتلكم, ضربني, فالح, سروال, حالين, كليه, بالر...       1570


## Results

Best C_V coherence: **6 topics** (CV 0.623, NPMI 0.549) — the highest C_V score of any combination tested in this project. Topic 4 (state/people/president/law themed) and Topic 2 (corruption/parliament/politics themed) read as the most internally consistent; Topic 5 mixes several unrelated threads, suggesting it may be absorbing "everything else" at 6 topics.

**Note:** in the original exploration notebook, the print statement in this cell reused a leftover `n_topics` variable from the grid search loop and printed "✅ 100 topics → ..." even though `n_components=6` was actually used — a cosmetic bug in the original code, corrected to `6 topics` in this cleaned-up version.